In [21]:
import numpy as np
preds = np.random.normal(loc=[0,0], scale=[1,2], size=(1000,2))
print(preds)
np.mean(preds[:,1])
ci_values = np.arange(0.1,1.1,0.1)

[[ 0.85378397  0.27405175]
 [-0.91649889  2.88937826]
 [-0.54516307 -0.39493213]
 ...
 [-2.07968292 -1.53433293]
 [ 0.02973454  0.87256401]
 [-0.06087516  0.67909482]]


In [16]:
def within_confidence_interval_xy(gt, pred, unc, confidence):
    """
    Determines whether bounding box position (x, y) is within given confidence interval.
    :param gt_box: GT annotation sample.
    :param pred_box: Predicted sample.
    :confidence: Confidence percentage in (0; 1.0)
    :distribution: Distribution that implements the percent point function (ppf) with mean 0 and variance 1.
        Default: stats.norm (i.e. Normal Gaussian)
    :return: Indicator 1 if position is within the confidence interval.
    """
    scale_factor = np.sqrt(-2 * np.log(1 - confidence))
    # z_score = distribution.ppf((1 + confidence) / 2)
    std_dev = np.sqrt(unc)
    if len(std_dev) == 0:
        return 0
    a = scale_factor * std_dev[0]
    b = scale_factor * std_dev[1]
    dist = np.abs(np.array(pred) - np.array(gt)) 
    if ((dist[0]/a)**2 + (dist[1]/b)**2 <= 1):
        return 1
    else:
        return 0

In [22]:
eval = {ci: [] for ci in ci_values}
for ci in ci_values:
    for pred in preds:
        eval[ci].append(within_confidence_interval_xy([0,0], pred, unc=[1,4], confidence=ci))


/tmp/ipykernel_5179/1815824444.py:11: RuntimeWarning: divide by zero encountered in log
  scale_factor = np.sqrt(-2 * np.log(1 - confidence))


In [23]:
for ci, values in eval.items():
    empirical_mean = np.mean(values)
    print("CI {0:2f}: {1:8.4f}".format(ci, empirical_mean))


CI 0.100000:   0.0980
CI 0.200000:   0.2060
CI 0.300000:   0.2940
CI 0.400000:   0.3850
CI 0.500000:   0.4830
CI 0.600000:   0.5710
CI 0.700000:   0.6810
CI 0.800000:   0.7950
CI 0.900000:   0.8990
CI 1.000000:   1.0000
